In [1]:
import boto3
import pandas as pd
import numpy as np

AWS_REGION = "ap-south-1"
BUCKET_NAME = "krushang-beverage-ml-2026"

RAW_KEY = "raw/survey_results.csv"
PROCESSED_PREFIX = "processed"

s3 = boto3.client("s3", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("S3 Bucket:", BUCKET_NAME)

AWS Region: ap-south-1
S3 Bucket: krushang-beverage-ml-2026


In [2]:
response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=RAW_KEY
)

df = pd.read_csv(response["Body"])

print("Raw dataset shape:", df.shape)
df.head(20)

Raw dataset shape: (30010, 17)


,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,NaN,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100
5,R00006,22,F,Urban,Student,NaN,5-7 times,Established,Large (1 L),2 to 4,Price,Traditional,Online,Simple,Low (Not very concerned),"Active (eg. Sports, gym)",100-150
6,R00007,45,F,Urban,Entrepreneur,10L - 15L,5-7 times,Newcomer,Medium (500 ml),2 to 4,Price,Traditional,Online,Premium,High (Very health-conscious),Social (eg. Parties),200-250
7,R00008,31,M,Urban,Entrepreneur,26L - 35L,5-7 times,Established,Medium (500 ml),above 4,Brand Reputation,Exotic,Retail Store,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",200-250
8,R00009,27,F,Semi-Urban,Working Professional,<10L,5-7 times,Newcomer,Small (250 ml),2 to 4,Availability,Exotic,Retail Store,Eco-Friendly,Medium (Moderately health-conscious),Social (eg. Parties),100-150
9,R00010,49,F,Semi-Urban,Entrepreneur,10L - 15L,5-7 times,Newcomer,Medium (500 ml),2 to 4,Price,Traditional,Retail Store,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",200-250


In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Shape: (30010, 17)

Columns:
['respondent_id', 'age', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'health_concerns', 'typical_consumption_situations', 'price_range']

Missing values:
respondent_id                        0
age                                  0
gender                               0
zone                                 0
occupation                           0
income_levels                     8064
consume_frequency(weekly)            8
current_brand                        0
preferable_consumption_size          0
awareness_of_other_brands            0
reasons_for_choosing_brands          0
flavor_preference                    0
purchase_channel                    10
packaging_preference                 0
health_concerns                      0
typical_consumption_situa

In [4]:
# Step 1: Remove exact duplicate rows

print("Shape before duplicate removal:", df.shape)
print("Exact duplicates:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after duplicate removal:", df.shape)

Shape before duplicate removal: (30010, 17)
Exact duplicates: 10
Shape after duplicate removal: (30000, 17)


In [5]:
# Step 2: Check categorical inconsistencies

for col in ["zone", "current_brand"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- zone ---
zone
Metro         11911
Urban         10688
Semi-Urban     5275
Rural          2117
urbna             5
Metor             4
Name: count, dtype: int64

--- current_brand ---
current_brand
Established    15447
Newcomer       14503
newcomer          30
Establishd        20
Name: count, dtype: int64


In [6]:
# Step 3: Fix confirmed categorical inconsistencies

df["zone"] = df["zone"].replace({
    "urbna": "Urban",
    "Metor": "Metro"
})

df["current_brand"] = df["current_brand"].replace({
    "newcomer": "Newcomer",
    "Establishd": "Established"
})

In [7]:
print(df["zone"].value_counts(dropna=False))
print()
print(df["current_brand"].value_counts(dropna=False))

zone
Metro         11915
Urban         10693
Semi-Urban     5275
Rural          2117
Name: count, dtype: int64

current_brand
Established    15467
Newcomer       14533
Name: count, dtype: int64


In [8]:
# Step 4: Handle deterministic missing values

df["income_levels"] = df["income_levels"].fillna("Not Reported")

print("Missing values after income handling:")
print(
    df[
        [
            "income_levels",
            "consume_frequency(weekly)",
            "purchase_channel"
        ]
    ].isnull().sum()
)

Missing values after income handling:
income_levels                 0
consume_frequency(weekly)     8
purchase_channel             10
dtype: int64


In [9]:
# Step 5: Check logical age / occupation anomalies

age_group_temp = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 70, float("inf")],
    labels=["18-25", "26-35", "36-45", "46-55", "56-70", "70+"]
)

print("Age range:", df["age"].min(), "to", df["age"].max())

print("\nStudents aged 56-70:")
print(
    df[
        (age_group_temp == "56-70") &
        (df["occupation"] == "Student")
    ][["respondent_id", "age", "occupation"]]
)

print("\nRespondents aged above 70:")
print(
    df[
        age_group_temp == "70+"
    ][["respondent_id", "age", "occupation"]]
)

print("\nTotal suspicious rows:",
      (
          ((age_group_temp == "56-70") & (df["occupation"] == "Student"))
          | (age_group_temp == "70+")
      ).sum()
)

Age range: 18 to 604

Students aged 56-70:
      respondent_id  age occupation
182          R00183   65    Student
3526         R03525   59    Student
3527         R03526   61    Student
3772         R03771   63    Student
4033         R04032   64    Student
6545         R06543   66    Student
6594         R06592   70    Student
6648         R06646   63    Student
7420         R07418   60    Student
7596         R07594   59    Student
7841         R07838   68    Student
9090         R09086   65    Student
9198         R09194   67    Student
11877        R11872   60    Student
12031        R12026   68    Student
12471        R12466   62    Student
15711        R15706   61    Student
16230        R16225   59    Student
16829        R16824   64    Student
17202        R17197   66    Student
19103        R19097   60    Student
20218        R20212   68    Student
21422        R21416   68    Student
24252        R24244   64    Student
24885        R24877   69    Student
25090        R25081  

In [10]:
# Step 6: Profile suspicious Student records

student_anomalies = df[
    (df["occupation"] == "Student") &
    (df["age"] > 35)
].copy()

student_anomalies[
    [
        "respondent_id",
        "age",
        "gender",
        "zone",
        "income_levels",
        "consume_frequency(weekly)",
        "current_brand",
        "preferable_consumption_size",
        "awareness_of_other_brands",
        "purchase_channel",
        "health_concerns"
    ]
]

,respondent_id,age,gender,zone,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,purchase_channel,health_concerns
182,R00183,65,F,Urban,Not Reported,5-7 times,Established,Large (1 L),above 4,Retail Store,Medium (Moderately health-conscious)
3526,R03525,59,F,Semi-Urban,Not Reported,0-2 times,Established,Medium (500 ml),0 to 1,Retail Store,Medium (Moderately health-conscious)
3527,R03526,61,M,Semi-Urban,Not Reported,0-2 times,Newcomer,Small (250 ml),2 to 4,Retail Store,High (Very health-conscious)
3772,R03771,63,M,Metro,Not Reported,5-7 times,Newcomer,Medium (500 ml),2 to 4,Online,High (Very health-conscious)
4033,R04032,64,M,Urban,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,Retail Store,High (Very health-conscious)
6545,R06543,66,F,Urban,Not Reported,5-7 times,Newcomer,Medium (500 ml),0 to 1,Retail Store,High (Very health-conscious)
6594,R06592,70,M,Semi-Urban,Not Reported,5-7 times,Established,Medium (500 ml),0 to 1,Retail Store,Medium (Moderately health-conscious)
6648,R06646,63,F,Semi-Urban,Not Reported,0-2 times,Established,Medium (500 ml),2 to 4,Retail Store,High (Very health-conscious)
7420,R07418,60,F,Rural,Not Reported,3-4 times,Established,Small (250 ml),0 to 1,Retail Store,Medium (Moderately health-conscious)
7596,R07594,59,M,Urban,Not Reported,5-7 times,Newcomer,Medium (500 ml),2 to 4,Online,High (Very health-conscious)


In [11]:
# Compare normal age patterns by occupation

df[df["age"] <= 70].groupby("occupation")["age"].agg(
    ["count", "min", "median", "mean", "max"]
).round(1)

,count,min,median,mean,max
occupation,,,,,
Entrepreneur,5000,18,36.0,37.4,70
Retired,1130,56,63.0,63.0,70
Student,8060,18,22.0,22.5,70
Working Professional,15801,18,34.0,34.7,70


In [12]:
# Income profile by occupation

pd.crosstab(
    df["occupation"],
    df["income_levels"],
    normalize="index"
).round(3)

income_levels,10L - 15L,16L - 25L,26L - 35L,<10L,> 35L,Not Reported
occupation,,,,,,
Entrepreneur,0.249,0.279,0.215,0.142,0.115,0.0
Retired,0.195,0.105,0.000,0.699,0.000,0.0
Student,0.000,0.000,0.000,0.000,0.000,1.0
Working Professional,0.239,0.278,0.177,0.200,0.106,0.0


In [13]:
# Create age groups

df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 25, 35, 45, 55, 70, float("inf")],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-70",
        "70+"
    ]
)

In [14]:
# Remove logically inconsistent / invalid age records

outlier_conditions = (
    ((df["age_group"] == "56-70") & (df["occupation"] == "Student"))
    |
    (df["age_group"] == "70+")
)

print("Rows to remove:", outlier_conditions.sum())

df = df.loc[~outlier_conditions].copy()

print("Final shape:", df.shape)

Rows to remove: 44
Final shape: (29956, 18)


In [15]:
# Remaining missing values

df["consume_frequency(weekly)"] = df["consume_frequency(weekly)"].fillna(
    df["consume_frequency(weekly)"].mode()[0]
)

df["purchase_channel"] = df["purchase_channel"].fillna(
    df["purchase_channel"].mode()[0]
)

print("Remaining missing values:", df.isnull().sum().sum())
print("Shape:", df.shape)

Remaining missing values: 0
Shape: (29956, 18)


In [16]:
zone_map = {
    "Rural": 1,
    "Semi-Urban": 2,
    "Urban": 3,
    "Metro": 4
}

income_map = {
    "Not Reported": 0,
    "<10L": 1,
    "10L - 15L": 2,
    "16L - 25L": 3,
    "26L - 35L": 4,
    "> 35L": 5
}

df["income_levels"] = df["income_levels"].str.strip()

df["zone_num"] = df["zone"].map(zone_map)
df["income_num"] = df["income_levels"].map(income_map)

df["zas_score"] = df["zone_num"] * df["income_num"]

In [17]:
df["bsi"] = (
    (df["current_brand"] != "Established") &
    (df["reasons_for_choosing_brands"].isin(["Price", "Quality"]))
).astype(int)

In [18]:
print(df[["age_group", "zone_num", "income_num", "zas_score", "bsi"]].head())

  age_group  zone_num  income_num  zas_score  bsi
0     26-35         3           1          3    1
1     46-55         4           5         20    0
2     36-45         1           5          5    0
3     26-35         3           3          9    0
4     18-25         4           0          0    0


In [19]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["price_range", "respondent_id"])
y = df["price_range"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))

X_train: (22467, 20)
X_test : (7489, 20)

Train target distribution:
price_range
200-250    0.324
150-200    0.294
100-150    0.260
50-100     0.122
Name: proportion, dtype: float64

Test target distribution:
price_range
200-250    0.324
150-200    0.294
100-150    0.260
50-100     0.122
Name: proportion, dtype: float64


In [20]:
from sklearn.impute import SimpleImputer

# Columns with missing values
impute_cols = [
    "consume_frequency(weekly)",
    "purchase_channel"
]

# Fit ONLY on training data
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train[impute_cols] = cat_imputer.fit_transform(
    X_train[impute_cols]
)

# Test only receives values learned from train
X_test[impute_cols] = cat_imputer.transform(
    X_test[impute_cols]
)

print("Learned training modes:")
for col, value in zip(impute_cols, cat_imputer.statistics_):
    print(f"{col}: {value}")

print("\nMissing train:", X_train[impute_cols].isnull().sum().sum())
print("Missing test :", X_test[impute_cols].isnull().sum().sum())

Learned training modes:
consume_frequency(weekly): 3-4 times
purchase_channel: Online

Missing train: 0
Missing test : 0


In [21]:
cf_map = {
    "0-2 times": 1,
    "3-4 times": 2,
    "5-7 times": 3
}

ab_map = {
    "0 to 1": 1,
    "2 to 4": 2,
    "above 4": 3
}

for data in [X_train, X_test]:
    data["cf_num"] = data["consume_frequency(weekly)"].map(cf_map)
    data["ab_num"] = data["awareness_of_other_brands"].map(ab_map)

    data["cf_ab_score"] = (
        data["cf_num"] /
        (data["cf_num"] + data["ab_num"])
    ).round(2)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nMissing values:")
print("Train:", X_train.isnull().sum().sum())
print("Test :", X_test.isnull().sum().sum())

Train shape: (22467, 23)
Test shape : (7489, 23)

Missing values:
Train: 0
Test : 0


In [22]:
# Step 9: Finalize model features

drop_cols = [
    "age",
    "cf_num",
    "ab_num",
    "zone_num",
    "income_num"
]

X_train = X_train.drop(columns=drop_cols)
X_test  = X_test.drop(columns=drop_cols)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (22467, 18)
Test : (7489, 18)


In [23]:
# Fixed ordinal mappings

age_group_map = {
    "18-25": 1,
    "26-35": 2,
    "36-45": 3,
    "46-55": 4,
    "56-70": 5
}

income_map = {
    "Not Reported": 0,
    "<10L": 1,
    "10L - 15L": 2,
    "16L - 25L": 3,
    "26L - 35L": 4,
    "> 35L": 5
}

health_map = {
    "Low (Not very concerned)": 1,
    "Medium (Moderately health-conscious)": 2,
    "High (Very health-conscious)": 3
}

freq_map = {
    "0-2 times": 1,
    "3-4 times": 2,
    "5-7 times": 3
}

size_map = {
    "Small (250 ml)": 1,
    "Medium (500 ml)": 2,
    "Large (1 L)": 3
}

for data in [X_train, X_test]:
    data["age_group"] = data["age_group"].astype(str).map(age_group_map)
    data["income_levels"] = data["income_levels"].map(income_map)
    data["health_concerns"] = data["health_concerns"].map(health_map)
    data["consume_frequency(weekly)"] = data["consume_frequency(weekly)"].map(freq_map)
    data["preferable_consumption_size"] = data["preferable_consumption_size"].map(size_map)

# Fixed target mapping
price_map = {
    "50-100": 0,
    "100-150": 1,
    "150-200": 2,
    "200-250": 3
}

y_train = y_train.map(price_map)
y_test  = y_test.map(price_map)

print("Missing train:", X_train.isna().sum().sum())
print("Missing test :", X_test.isna().sum().sum())
print("Train shape:", X_train.shape)

Missing train: 0
Missing test : 0
Train shape: (22467, 18)


In [24]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Remaining nominal categorical columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:", cat_cols)

# Fit encoder ONLY on training data
ohe = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

X_train_cat = ohe.fit_transform(X_train[cat_cols])
X_test_cat  = ohe.transform(X_test[cat_cols])

# Convert encoded arrays to DataFrames
encoded_cols = ohe.get_feature_names_out(cat_cols)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_cols,
    index=X_test.index
)

# Numeric features pass through unchanged
num_cols = X_train.columns.difference(cat_cols)

X_train_encoded = pd.concat(
    [X_train[num_cols], X_train_cat],
    axis=1
)

X_test_encoded = pd.concat(
    [X_test[num_cols], X_test_cat],
    axis=1
)

print("Encoded train:", X_train_encoded.shape)
print("Encoded test :", X_test_encoded.shape)

print("\nColumns identical:",
      X_train_encoded.columns.equals(X_test_encoded.columns))

print("Missing train:", X_train_encoded.isna().sum().sum())
print("Missing test :", X_test_encoded.isna().sum().sum())

Categorical columns: ['gender', 'zone', 'occupation', 'current_brand', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel', 'packaging_preference', 'typical_consumption_situations']
Encoded train: (22467, 27)
Encoded test : (7489, 27)

Columns identical: True
Missing train: 0
Missing test : 0


In [25]:
from sklearn.model_selection import train_test_split

# Recreate the same untouched development/test split
X_dev_raw, X_final_test_raw, y_dev, y_final_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Validation comes ONLY from the development data
X_model_train, X_val, y_model_train, y_val = train_test_split(
    X_dev_raw,
    y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

print("Model train :", X_model_train.shape)
print("Validation  :", X_val.shape)
print("Final test  :", X_final_test_raw.shape)

print("\nOverall split:")
print(
    round(len(X_model_train)/len(X)*100, 1),
    round(len(X_val)/len(X)*100, 1),
    round(len(X_final_test_raw)/len(X)*100, 1)
)

Model train : (17973, 20)
Validation  : (4494, 20)
Final test  : (7489, 20)

Overall split:
60.0 15.0 25.0


In [26]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# --------------------------------------------------
# STEP 11: Fit preprocessing on MODEL TRAIN only
# --------------------------------------------------

train_prep = X_model_train.copy()
val_prep   = X_val.copy()

# 1. Train-only imputation
impute_cols = [
    "consume_frequency(weekly)",
    "purchase_channel"
]

imputer = SimpleImputer(strategy="most_frequent")

train_prep[impute_cols] = imputer.fit_transform(
    train_prep[impute_cols]
)

val_prep[impute_cols] = imputer.transform(
    val_prep[impute_cols]
)


# 2. Create cf_ab_score AFTER imputation
cf_map = {
    "0-2 times": 1,
    "3-4 times": 2,
    "5-7 times": 3
}

ab_map = {
    "0 to 1": 1,
    "2 to 4": 2,
    "above 4": 3
}

for data in [train_prep, val_prep]:
    data["cf_num"] = data["consume_frequency(weekly)"].map(cf_map)
    data["ab_num"] = data["awareness_of_other_brands"].map(ab_map)

    data["cf_ab_score"] = (
        data["cf_num"] /
        (data["cf_num"] + data["ab_num"])
    ).round(2)


# 3. Remove raw/helper features not used by final model
drop_cols = [
    "age",
    "cf_num",
    "ab_num",
    "zone_num",
    "income_num"
]

train_prep = train_prep.drop(columns=drop_cols)
val_prep   = val_prep.drop(columns=drop_cols)


# 4. Fixed ordinal mappings
age_group_map = {
    "18-25": 1,
    "26-35": 2,
    "36-45": 3,
    "46-55": 4,
    "56-70": 5
}

income_map = {
    "Not Reported": 0,
    "<10L": 1,
    "10L - 15L": 2,
    "16L - 25L": 3,
    "26L - 35L": 4,
    "> 35L": 5
}

health_map = {
    "Low (Not very concerned)": 1,
    "Medium (Moderately health-conscious)": 2,
    "High (Very health-conscious)": 3
}

freq_map = {
    "0-2 times": 1,
    "3-4 times": 2,
    "5-7 times": 3
}

size_map = {
    "Small (250 ml)": 1,
    "Medium (500 ml)": 2,
    "Large (1 L)": 3
}

for data in [train_prep, val_prep]:
    data["age_group"] = data["age_group"].astype(str).map(age_group_map)
    data["income_levels"] = data["income_levels"].map(income_map)
    data["health_concerns"] = data["health_concerns"].map(health_map)
    data["consume_frequency(weekly)"] = data["consume_frequency(weekly)"].map(freq_map)
    data["preferable_consumption_size"] = data["preferable_consumption_size"].map(size_map)


# 5. One-hot encoder fitted ONLY on model train
cat_cols = train_prep.select_dtypes(
    include=["object", "category"]
).columns.tolist()

num_cols = [
    c for c in train_prep.columns
    if c not in cat_cols
]

ohe = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

train_cat = ohe.fit_transform(train_prep[cat_cols])
val_cat   = ohe.transform(val_prep[cat_cols])

encoded_cols = ohe.get_feature_names_out(cat_cols)

train_cat = pd.DataFrame(
    train_cat,
    columns=encoded_cols,
    index=train_prep.index
)

val_cat = pd.DataFrame(
    val_cat,
    columns=encoded_cols,
    index=val_prep.index
)

X_model_train_encoded = pd.concat(
    [train_prep[num_cols], train_cat],
    axis=1
)

X_val_encoded = pd.concat(
    [val_prep[num_cols], val_cat],
    axis=1
)


# 6. Encode target
price_map = {
    "50-100": 0,
    "100-150": 1,
    "150-200": 2,
    "200-250": 3
}

y_model_train_encoded = y_model_train.map(price_map)
y_val_encoded = y_val.map(price_map)


# Validation
print("Model train:", X_model_train_encoded.shape)
print("Validation :", X_val_encoded.shape)

print("Same columns:",
      X_model_train_encoded.columns.equals(X_val_encoded.columns))

print("Train missing:", X_model_train_encoded.isna().sum().sum())
print("Val missing  :", X_val_encoded.isna().sum().sum())

print("\nImputer learned:")
for col, value in zip(impute_cols, imputer.statistics_):
    print(f"{col}: {value}")

Model train: (17973, 27)
Validation : (4494, 27)
Same columns: True
Train missing: 0
Val missing  : 0

Imputer learned:
consume_frequency(weekly): 3-4 times
purchase_channel: Online


In [27]:
# STEP 12: Baseline model comparison

import time
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# --------------------------------------------------
# Scale ONLY for Logistic Regression
# --------------------------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_model_train_encoded)
X_val_scaled = scaler.transform(X_val_encoded)


# --------------------------------------------------
# Baseline models
# --------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
}


results = []

for name, model in models.items():

    start = time.time()

    # Logistic uses scaled data
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_model_train_encoded)
        preds = model.predict(X_val_scaled)

    # Tree models use original encoded data
    else:
        model.fit(
            X_model_train_encoded,
            y_model_train_encoded
        )
        preds = model.predict(X_val_encoded)

    elapsed = time.time() - start

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val_encoded, preds),
        "Precision_Macro": precision_score(
            y_val_encoded, preds, average="macro"
        ),
        "Recall_Macro": recall_score(
            y_val_encoded, preds, average="macro"
        ),
        "F1_Macro": f1_score(
            y_val_encoded, preds, average="macro"
        ),
        "F1_Weighted": f1_score(
            y_val_encoded, preds, average="weighted"
        ),
        "Training_Time_sec": elapsed
    })


baseline_results = pd.DataFrame(results)

baseline_results = baseline_results.sort_values(
    "F1_Macro",
    ascending=False
).reset_index(drop=True)

baseline_results.round(4)

,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,F1_Weighted,Training_Time_sec
0,XGBoost,0.9190,0.9191,0.9159,0.9175,0.9190,1.3401
1,LightGBM,0.9179,0.9175,0.9149,0.9162,0.9179,0.8359
2,Random Forest,0.8901,0.8932,0.8861,0.8894,0.8903,3.9987
3,Logistic Regression,0.8498,0.8465,0.8425,0.8445,0.8498,0.3266


In [28]:
# STEP 13: Per-class validation performance

from sklearn.metrics import classification_report

class_names = ["50-100", "100-150", "150-200", "200-250"]

validation_predictions = {}

for name, model in models.items():

    if name == "Logistic Regression":
        preds = model.predict(X_val_scaled)
    else:
        preds = model.predict(X_val_encoded)

    validation_predictions[name] = preds

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            y_val_encoded,
            preds,
            target_names=class_names,
            digits=4
        )
    )


Logistic Regression
              precision    recall  f1-score   support

      50-100     0.8469    0.8175    0.8319       548
     100-150     0.8092    0.8238    0.8164      1169
     150-200     0.8148    0.8098    0.8123      1320
     200-250     0.9152    0.9190    0.9171      1457

    accuracy                         0.8498      4494
   macro avg     0.8465    0.8425    0.8445      4494
weighted avg     0.8498    0.8498    0.8498      4494


Random Forest
              precision    recall  f1-score   support

      50-100     0.9178    0.8759    0.8964       548
     100-150     0.8762    0.8597    0.8679      1169
     150-200     0.8397    0.8773    0.8581      1320
     200-250     0.9391    0.9314    0.9352      1457

    accuracy                         0.8901      4494
   macro avg     0.8932    0.8861    0.8894      4494
weighted avg     0.8910    0.8901    0.8903      4494


XGBoost
              precision    recall  f1-score   support

      50-100     0.9290    0.9

In [29]:
from sklearn.metrics import confusion_matrix

for name, preds in validation_predictions.items():

    cm = confusion_matrix(y_val_encoded, preds)

    cm_df = pd.DataFrame(
        cm,
        index=[f"Actual_{c}" for c in class_names],
        columns=[f"Pred_{c}" for c in class_names]
    )

    print(f"\n{name}")
    display(cm_df)


Logistic Regression


,Pred_50-100,Pred_100-150,Pred_150-200,Pred_200-250
Actual_50-100,448,100,0,0
Actual_100-150,81,963,125,0
Actual_150-200,0,127,1069,124
Actual_200-250,0,0,118,1339



Random Forest


,Pred_50-100,Pred_100-150,Pred_150-200,Pred_200-250
Actual_50-100,480,68,0,0
Actual_100-150,43,1005,121,0
Actual_150-200,0,74,1158,88
Actual_200-250,0,0,100,1357



XGBoost


,Pred_50-100,Pred_100-150,Pred_150-200,Pred_200-250
Actual_50-100,497,51,0,0
Actual_100-150,38,1055,76,0
Actual_150-200,0,65,1188,67
Actual_200-250,0,0,67,1390



LightGBM


,Pred_50-100,Pred_100-150,Pred_150-200,Pred_200-250
Actual_50-100,498,50,0,0
Actual_100-150,41,1047,81,0
Actual_150-200,0,63,1188,69
Actual_200-250,0,0,65,1392


In [31]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

In [32]:
def objective_xgb(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 800
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 8
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 8
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 0.5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 1.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 10.0, log=True
        ),

        "random_state": 42,
        "n_jobs": -1,
        "eval_metric": "mlogloss",
        "verbosity": 0
    }

    model = XGBClassifier(**params)

    model.fit(
        X_model_train_encoded,
        y_model_train_encoded
    )

    preds = model.predict(X_val_encoded)

    return f1_score(
        y_val_encoded,
        preds,
        average="macro"
    )

In [33]:
study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_xgb.optimize(
    objective_xgb,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-09-03 14:15:12,763] A new study created in memory with name: no-name-0541cf6e-de9c-41b9-8354-8e0cc35e3d59


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-03 14:15:19,196] Trial 0 finished with value: 0.916001316700431 and parameters: {'n_estimators': 425, 'max_depth': 8, 'learning_rate': 0.07259248719561363, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309, 'min_child_weight': 2, 'gamma': 0.02904180608409973, 'reg_alpha': 0.29154431891537513, 'reg_lambda': 1.5930522616241019}. Best is trial 0 with value: 0.916001316700431.
[I 2026-09-03 14:15:24,216] Trial 1 finished with value: 0.9145467403062613 and parameters: {'n_estimators': 625, 'max_depth': 3, 'learning_rate': 0.13826189316223852, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'min_child_weight': 2, 'gamma': 0.09170225492671691, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 1.1207606211860568}. Best is trial 0 with value: 0.916001316700431.
[I 2026-09-03 14:15:28,373] Trial 2 finished with value: 0.906960848774214 and parameters: {'n_estimators': 459, 'max_depth': 4, 'learning_rate': 0.05243180891902853, 'subsample':

In [34]:
print("Best Macro F1:", study_xgb.best_value)

print("\nBest parameters:")
for k, v in study_xgb.best_params.items():
    print(f"{k}: {v}")

Best Macro F1: 0.9230423603152382

Best parameters:
n_estimators: 700
max_depth: 6
learning_rate: 0.12034778187740461
subsample: 0.9350139863013668
colsample_bytree: 0.9552153727216537
min_child_weight: 5
gamma: 0.13876309692419167
reg_alpha: 0.03870751721089646
reg_lambda: 1.9028977942385588


In [35]:
xgb_trials = study_xgb.trials_dataframe()

xgb_trials[
    [
        "number",
        "value",
        "params_n_estimators",
        "params_max_depth",
        "params_learning_rate",
        "params_subsample",
        "params_colsample_bytree",
        "params_min_child_weight",
        "params_gamma",
        "params_reg_alpha",
        "params_reg_lambda"
    ]
].sort_values(
    "value",
    ascending=False
).head(10)

,number,value,params_n_estimators,params_max_depth,params_learning_rate,params_subsample,params_colsample_bytree,params_min_child_weight,params_gamma,params_reg_alpha,params_reg_lambda
33,33,0.923042,700,6,0.120348,0.935014,0.955215,5,0.138763,0.038708,1.902898
31,31,0.923015,764,6,0.124409,0.962465,0.926778,6,0.148487,0.014273,2.543923
17,17,0.922632,709,7,0.111619,0.909804,0.906902,5,0.143067,0.005882,9.772041
21,21,0.922343,722,7,0.131779,0.968254,0.841036,5,0.136195,0.043791,2.667592
25,25,0.922191,718,7,0.113963,0.938507,0.954908,6,0.207450,0.003374,2.088044
48,48,0.922087,748,7,0.119924,0.979189,0.865049,8,0.206422,0.005647,1.247433
13,13,0.922069,798,6,0.141938,0.999809,0.848924,6,0.120777,0.017396,2.469330
32,32,0.922027,759,7,0.116116,0.953268,0.921895,6,0.207497,0.152683,1.190105
41,41,0.921633,709,7,0.106949,0.938200,0.957469,6,0.196908,0.002173,1.968677
26,26,0.921430,720,8,0.081308,0.931889,0.957577,4,0.251006,0.002020,7.517850


In [36]:
# STEP 15: LightGBM Optuna tuning

import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score


def objective_lgbm(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 900
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves", 20, 100
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 80
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.7, 1.0
        ),

        # Important: otherwise LightGBM may not actually use subsample
        "subsample_freq": 1,

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.7, 1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 1.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 0.1, 10.0, log=True
        ),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_model_train_encoded,
        y_model_train_encoded
    )

    preds = model.predict(X_val_encoded)

    return f1_score(
        y_val_encoded,
        preds,
        average="macro"
    )


study_lgbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_lgbm.optimize(
    objective_lgbm,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-09-03 14:28:27,235] A new study created in memory with name: no-name-e2de7afc-cf2b-4adc-a6b2-69f58cb6928e


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-03 14:28:32,148] Trial 0 finished with value: 0.9158874372566668 and parameters: {'n_estimators': 462, 'learning_rate': 0.13125830316209655, 'num_leaves': 79, 'max_depth': 7, 'min_child_samples': 21, 'subsample': 0.7467983561008608, 'colsample_bytree': 0.7174250836504598, 'reg_alpha': 0.29154431891537513, 'reg_lambda': 1.5930522616241019}. Best is trial 0 with value: 0.9158874372566668.
[I 2026-09-03 14:28:41,551] Trial 1 finished with value: 0.9109753662771625 and parameters: {'n_estimators': 696, 'learning_rate': 0.010573268083515799, 'num_leaves': 98, 'max_depth': 9, 'min_child_samples': 25, 'subsample': 0.7545474901621302, 'colsample_bytree': 0.7550213529560301, 'reg_alpha': 0.0016480446427978971, 'reg_lambda': 1.1207606211860568}. Best is trial 0 with value: 0.9158874372566668.
[I 2026-09-03 14:28:44,813] Trial 2 finished with value: 0.897942641401612 and parameters: {'n_estimators': 502, 'learning_rate': 0.022004527434741072, 'num_leaves': 69, 'max_depth': 4, 'min_chil

In [37]:
print("Best LightGBM Macro F1:", study_lgbm.best_value)

print("\nBest parameters:")
for k, v in study_lgbm.best_params.items():
    print(f"{k}: {v}")

Best LightGBM Macro F1: 0.9223663833968606

Best parameters:
n_estimators: 752
learning_rate: 0.12741535065931847
num_leaves: 95
max_depth: 4
min_child_samples: 68
subsample: 0.9784078582833712
colsample_bytree: 0.7510703040726703
reg_alpha: 0.0036519195859391557
reg_lambda: 0.37782097364722056


In [38]:
lgbm_trials = study_lgbm.trials_dataframe()

lgbm_trials[
    [
        "number",
        "value",
        "params_n_estimators",
        "params_learning_rate",
        "params_num_leaves",
        "params_max_depth",
        "params_min_child_samples",
        "params_subsample",
        "params_colsample_bytree",
        "params_reg_alpha",
        "params_reg_lambda"
    ]
].sort_values(
    "value",
    ascending=False
).head(10)

,number,value,params_n_estimators,params_learning_rate,params_num_leaves,params_max_depth,params_min_child_samples,params_subsample,params_colsample_bytree,params_reg_alpha,params_reg_lambda
32,32,0.922366,752,0.127415,95,4,68,0.978408,0.751070,0.003652,0.377821
29,29,0.922196,896,0.126382,44,4,80,0.976873,0.753991,0.007692,0.669215
34,34,0.922014,758,0.094346,57,4,75,0.978022,0.735153,0.005755,0.395018
25,25,0.921995,823,0.079360,29,4,71,0.962241,0.813255,0.004767,0.927878
31,31,0.921809,783,0.116345,38,4,80,0.974030,0.802091,0.003753,0.689382
21,21,0.921717,859,0.108953,33,4,79,0.989010,0.855186,0.047930,1.706717
46,46,0.921642,792,0.144520,27,4,66,0.978841,0.757621,0.025934,0.106941
23,23,0.921200,833,0.112724,35,4,78,0.962445,0.796127,0.035580,0.594935
41,41,0.921175,794,0.124774,93,4,76,0.975729,0.751457,0.004627,0.702661
40,40,0.921148,787,0.071529,65,4,68,0.948481,0.774494,0.010914,0.324335


In [39]:
# STEP 16: Random Forest Optuna tuning

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import optuna


def objective_rf(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 800
        ),

        "max_depth": trial.suggest_categorical(
            "max_depth", [None, 8, 10, 12, 15, 20, 25]
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 10
        ),

        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", 0.5, 0.75]
        ),

        "class_weight": trial.suggest_categorical(
            "class_weight",
            [None, "balanced", "balanced_subsample"]
        ),

        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            [True, False]
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(**params)

    model.fit(
        X_model_train_encoded,
        y_model_train_encoded
    )

    preds = model.predict(X_val_encoded)

    return f1_score(
        y_val_encoded,
        preds,
        average="macro"
    )


study_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_rf.optimize(
    objective_rf,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-09-03 14:34:56,182] A new study created in memory with name: no-name-b0e8576f-6430-46df-80b4-4b92d636cfb6


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-03 14:35:01,148] Trial 0 finished with value: 0.8539401091875066 and parameters: {'n_estimators': 425, 'max_depth': None, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'log2', 'class_weight': 'balanced_subsample', 'bootstrap': True}. Best is trial 0 with value: 0.8539401091875066.
[I 2026-09-03 14:35:04,573] Trial 1 finished with value: 0.866099548618144 and parameters: {'n_estimators': 375, 'max_depth': 20, 'min_samples_split': 11, 'min_samples_leaf': 6, 'max_features': 'log2', 'class_weight': 'balanced', 'bootstrap': True}. Best is trial 1 with value: 0.866099548618144.
[I 2026-09-03 14:35:14,360] Trial 2 finished with value: 0.8968709185103885 and parameters: {'n_estimators': 611, 'max_depth': 15, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': 0.5, 'class_weight': None, 'bootstrap': True}. Best is trial 2 with value: 0.8968709185103885.
[I 2026-09-03 14:35:21,444] Trial 3 finished with value: 0.9088097206006163 and parameters: {'n_estimat

In [40]:
print("Best RF Macro F1:", study_rf.best_value)

print("\nBest parameters:")
for k, v in study_rf.best_params.items():
    print(f"{k}: {v}")

Best RF Macro F1: 0.9132078845346865

Best parameters:
n_estimators: 302
max_depth: 15
min_samples_split: 2
min_samples_leaf: 3
max_features: 0.5
class_weight: balanced_subsample
bootstrap: False


In [41]:
rf_trials = study_rf.trials_dataframe()

rf_trials[
    [
        "number",
        "value",
        "params_n_estimators",
        "params_max_depth",
        "params_min_samples_split",
        "params_min_samples_leaf",
        "params_max_features",
        "params_class_weight",
        "params_bootstrap"
    ]
].sort_values(
    "value",
    ascending=False
).head(10)

,number,value,params_n_estimators,params_max_depth,params_min_samples_split,params_min_samples_leaf,params_max_features,params_class_weight,params_bootstrap
14,14,0.913208,302,15.0,2,3,0.5,balanced_subsample,False
27,27,0.912391,402,15.0,2,2,0.5,balanced_subsample,False
44,44,0.912154,395,20.0,5,4,0.5,balanced,False
36,36,0.912112,238,15.0,3,3,0.5,balanced,False
47,47,0.911978,387,20.0,8,4,0.5,balanced,False
43,43,0.911661,381,20.0,4,4,0.5,balanced,False
32,32,0.911007,350,15.0,4,1,0.5,balanced_subsample,False
18,18,0.910851,350,NaN,2,4,0.5,balanced_subsample,False
33,33,0.910834,349,15.0,4,1,0.5,balanced_subsample,False
24,24,0.910752,335,15.0,9,2,0.5,balanced_subsample,False


In [42]:
# STEP 17: Logistic Regression Optuna tuning

import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def objective_lr(trial):

    solver = trial.suggest_categorical(
        "solver",
        ["lbfgs", "saga"]
    )

    # Valid penalty choices depend on solver
    if solver == "lbfgs":
        penalty = "l2"
    else:
        penalty = trial.suggest_categorical(
            "penalty",
            ["l1", "l2", "elasticnet"]
        )

    params = {
        "C": trial.suggest_float(
            "C", 1e-3, 100, log=True
        ),

        "solver": solver,

        "penalty": penalty,

        "class_weight": trial.suggest_categorical(
            "class_weight",
            [None, "balanced"]
        ),

        "max_iter": 3000,
        "random_state": 42
    }

    # Only elasticnet requires l1_ratio
    if penalty == "elasticnet":
        params["l1_ratio"] = trial.suggest_float(
            "l1_ratio", 0.0, 1.0
        )

    model = LogisticRegression(**params)

    model.fit(
        X_train_scaled,
        y_model_train_encoded
    )

    preds = model.predict(X_val_scaled)

    return f1_score(
        y_val_encoded,
        preds,
        average="macro"
    )


study_lr = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study_lr.optimize(
    objective_lr,
    n_trials=50,
    show_progress_bar=True
)

[I 2026-09-03 14:43:40,348] A new study created in memory with name: no-name-2edb2ea3-3a68-4815-9a87-3ac1a731db74


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-09-03 14:43:40,748] Trial 0 finished with value: 0.7972449751567905 and parameters: {'solver': 'saga', 'penalty': 'l1', 'C': 0.0060252157362038605, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7972449751567905.
[I 2026-09-03 14:43:41,049] Trial 1 finished with value: 0.8219267910154431 and parameters: {'solver': 'saga', 'penalty': 'l2', 'C': 0.011526449540315618, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8219267910154431.
[I 2026-09-03 14:43:41,432] Trial 2 finished with value: 0.8061408513082259 and parameters: {'solver': 'saga', 'penalty': 'elasticnet', 'C': 0.004982752357076452, 'class_weight': 'balanced', 'l1_ratio': 0.45606998421703593}. Best is trial 1 with value: 0.8219267910154431.
[I 2026-09-03 14:43:41,709] Trial 3 finished with value: 0.8449254955582788 and parameters: {'solver': 'lbfgs', 'C': 0.37253938395788866, 'class_weight': None}. Best is trial 3 with value: 0.8449254955582788.
[I 2026-09-03 14:43:41,786] Trial 4 finished with val

In [43]:
print("Best Logistic Regression Macro F1:", study_lr.best_value)

print("\nBest parameters:")
for k, v in study_lr.best_params.items():
    print(f"{k}: {v}")

Best Logistic Regression Macro F1: 0.8454434692331583

Best parameters:
solver: lbfgs
C: 0.355183391139837
class_weight: None


In [44]:
lr_trials = study_lr.trials_dataframe()

cols = [
    "number",
    "value",
    "params_C",
    "params_solver",
    "params_penalty",
    "params_class_weight"
]

if "params_l1_ratio" in lr_trials.columns:
    cols.append("params_l1_ratio")

lr_trials[cols].sort_values(
    "value",
    ascending=False
).head(10)

,number,value,params_C,params_solver,params_penalty,params_class_weight,params_l1_ratio
31,31,0.845443,0.355183,lbfgs,NaN,None,NaN
17,17,0.845080,0.397613,lbfgs,NaN,None,NaN
21,21,0.845080,0.404431,lbfgs,NaN,None,NaN
3,3,0.844925,0.372539,lbfgs,NaN,None,NaN
42,42,0.844886,0.364765,lbfgs,NaN,None,NaN
11,11,0.844875,0.346060,lbfgs,NaN,None,NaN
18,18,0.844834,0.879722,lbfgs,NaN,None,NaN
39,39,0.844762,1.554610,lbfgs,NaN,None,NaN
32,32,0.844643,1.110591,lbfgs,NaN,None,NaN
15,15,0.844595,7.344283,lbfgs,NaN,None,NaN


In [45]:
# STEP 18: Final tuned validation comparison

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import pandas as pd


# -----------------------------
# Rebuild tuned models
# -----------------------------

best_xgb = XGBClassifier(
    **study_xgb.best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    verbosity=0
)

best_lgbm = LGBMClassifier(
    **study_lgbm.best_params,
    subsample_freq=1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

best_rf = RandomForestClassifier(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)


# LR needs conditional penalty reconstructed
lr_params = study_lr.best_params.copy()

if lr_params["solver"] == "lbfgs":
    lr_params["penalty"] = "l2"

best_lr = LogisticRegression(
    **lr_params,
    max_iter=3000,
    random_state=42
)


# -----------------------------
# Train + validation prediction
# -----------------------------

best_xgb.fit(
    X_model_train_encoded,
    y_model_train_encoded
)

best_lgbm.fit(
    X_model_train_encoded,
    y_model_train_encoded
)

best_rf.fit(
    X_model_train_encoded,
    y_model_train_encoded
)

best_lr.fit(
    X_train_scaled,
    y_model_train_encoded
)


tuned_predictions = {
    "XGBoost": best_xgb.predict(X_val_encoded),
    "LightGBM": best_lgbm.predict(X_val_encoded),
    "Random Forest": best_rf.predict(X_val_encoded),
    "Logistic Regression": best_lr.predict(X_val_scaled)
}


# -----------------------------
# Final validation metrics
# -----------------------------

results = []

for name, preds in tuned_predictions.items():

    # Since target classes are ordered 0,1,2,3:
    ordinal_mae = abs(
        y_val_encoded.to_numpy() - preds
    ).mean()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            y_val_encoded, preds
        ),
        "Precision_Macro": precision_score(
            y_val_encoded, preds, average="macro"
        ),
        "Recall_Macro": recall_score(
            y_val_encoded, preds, average="macro"
        ),
        "F1_Macro": f1_score(
            y_val_encoded, preds, average="macro"
        ),
        "F1_Weighted": f1_score(
            y_val_encoded, preds, average="weighted"
        ),
        "Ordinal_MAE": ordinal_mae
    })


tuned_leaderboard = pd.DataFrame(results).sort_values(
    ["F1_Macro", "Ordinal_MAE"],
    ascending=[False, True]
).reset_index(drop=True)

tuned_leaderboard.round(4)

,Model,Accuracy,Precision_Macro,Recall_Macro,F1_Macro,F1_Weighted,Ordinal_MAE
0,XGBoost,0.9250,0.9248,0.9215,0.9230,0.9250,0.0750
1,LightGBM,0.9246,0.9235,0.9213,0.9224,0.9246,0.0754
2,Random Forest,0.9126,0.9109,0.9161,0.9132,0.9125,0.0874
3,Logistic Regression,0.8505,0.8475,0.8435,0.8454,0.8505,0.1495


In [46]:
# Copies: final test remains untouched until transformed here
dev_prep = X_dev_raw.copy()
test_prep = X_final_test_raw.copy()

# --------------------------------------------------
# 1. Imputation fitted ONLY on full development set
# --------------------------------------------------

final_imputer = SimpleImputer(strategy="most_frequent")

dev_prep[impute_cols] = final_imputer.fit_transform(
    dev_prep[impute_cols]
)

test_prep[impute_cols] = final_imputer.transform(
    test_prep[impute_cols]
)


# --------------------------------------------------
# 2. cf_ab_score
# --------------------------------------------------

for data in [dev_prep, test_prep]:

    data["cf_num"] = data["consume_frequency(weekly)"].map(cf_map)
    data["ab_num"] = data["awareness_of_other_brands"].map(ab_map)

    data["cf_ab_score"] = (
        data["cf_num"] /
        (data["cf_num"] + data["ab_num"])
    ).round(2)


# --------------------------------------------------
# 3. Drop raw/helper columns
# --------------------------------------------------

drop_cols = [
    "age",
    "cf_num",
    "ab_num",
    "zone_num",
    "income_num"
]

dev_prep = dev_prep.drop(columns=drop_cols)
test_prep = test_prep.drop(columns=drop_cols)


# --------------------------------------------------
# 4. Fixed ordinal mappings
# --------------------------------------------------

for data in [dev_prep, test_prep]:

    data["age_group"] = (
        data["age_group"].astype(str).map(age_group_map)
    )

    data["income_levels"] = (
        data["income_levels"].map(income_map)
    )

    data["health_concerns"] = (
        data["health_concerns"].map(health_map)
    )

    data["consume_frequency(weekly)"] = (
        data["consume_frequency(weekly)"].map(freq_map)
    )

    data["preferable_consumption_size"] = (
        data["preferable_consumption_size"].map(size_map)
    )


# --------------------------------------------------
# 5. OHE fitted ONLY on development set
# --------------------------------------------------

final_cat_cols = dev_prep.select_dtypes(
    include=["object", "category"]
).columns.tolist()

final_num_cols = [
    c for c in dev_prep.columns
    if c not in final_cat_cols
]

final_ohe = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)

dev_cat = final_ohe.fit_transform(
    dev_prep[final_cat_cols]
)

test_cat = final_ohe.transform(
    test_prep[final_cat_cols]
)

encoded_cols = final_ohe.get_feature_names_out(
    final_cat_cols
)

dev_cat = pd.DataFrame(
    dev_cat,
    columns=encoded_cols,
    index=dev_prep.index
)

test_cat = pd.DataFrame(
    test_cat,
    columns=encoded_cols,
    index=test_prep.index
)

X_dev_final = pd.concat(
    [dev_prep[final_num_cols], dev_cat],
    axis=1
)

X_test_final = pd.concat(
    [test_prep[final_num_cols], test_cat],
    axis=1
)


# Target
y_dev_final = y_dev.map(price_map)
y_test_final = y_final_test.map(price_map)


print("Development:", X_dev_final.shape)
print("Final test :", X_test_final.shape)

print(
    "Same columns:",
    X_dev_final.columns.equals(X_test_final.columns)
)

print("Dev missing :", X_dev_final.isna().sum().sum())
print("Test missing:", X_test_final.isna().sum().sum())

Development: (22467, 27)
Final test : (7489, 27)
Same columns: True
Dev missing : 0
Test missing: 0


In [47]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

final_xgb = XGBClassifier(
    **study_xgb.best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss",
    verbosity=0
)

# Train on ALL development data
final_xgb.fit(
    X_dev_final,
    y_dev_final
)

# FIRST AND FINAL prediction on holdout test
final_test_pred = final_xgb.predict(
    X_test_final
)


final_metrics = {
    "Accuracy": accuracy_score(
        y_test_final, final_test_pred
    ),
    "Precision_Macro": precision_score(
        y_test_final, final_test_pred, average="macro"
    ),
    "Recall_Macro": recall_score(
        y_test_final, final_test_pred, average="macro"
    ),
    "F1_Macro": f1_score(
        y_test_final, final_test_pred, average="macro"
    ),
    "F1_Weighted": f1_score(
        y_test_final, final_test_pred, average="weighted"
    ),
    "Ordinal_MAE": abs(
        y_test_final.to_numpy() - final_test_pred
    ).mean()
}

pd.Series(final_metrics).round(4)

Accuracy           0.9246
Precision_Macro    0.9237
Recall_Macro       0.9235
F1_Macro           0.9236
F1_Weighted        0.9247
Ordinal_MAE        0.0754
dtype: float64

In [48]:
print(
    classification_report(
        y_test_final,
        final_test_pred,
        target_names=[
            "50-100",
            "100-150",
            "150-200",
            "200-250"
        ],
        digits=4
    )
)

cm = confusion_matrix(
    y_test_final,
    final_test_pred
)

pd.DataFrame(
    cm,
    index=[
        "Actual_50-100",
        "Actual_100-150",
        "Actual_150-200",
        "Actual_200-250"
    ],
    columns=[
        "Pred_50-100",
        "Pred_100-150",
        "Pred_150-200",
        "Pred_200-250"
    ]
)

              precision    recall  f1-score   support

      50-100     0.9265    0.9245    0.9255       914
     100-150     0.9110    0.9091    0.9101      1948
     150-200     0.8958    0.9109    0.9033      2199
     200-250     0.9616    0.9493    0.9554      2428

    accuracy                         0.9246      7489
   macro avg     0.9237    0.9235    0.9236      7489
weighted avg     0.9248    0.9246    0.9247      7489



,Pred_50-100,Pred_100-150,Pred_150-200,Pred_200-250
Actual_50-100,845,69,0,0
Actual_100-150,67,1771,110,0
Actual_150-200,0,104,2003,92
Actual_200-250,0,0,123,2305


In [49]:
# STEP 21: Persist final benchmark artifacts to S3

import boto3
import json
import pandas as pd

s3 = boto3.client("s3")

bucket = "krushang-beverage-ml-2026"


# -----------------------------
# 1. Final evaluation metrics
# -----------------------------

evaluation_summary = {
    "selected_model": "XGBoost",
    "model_selection_metric": "macro_f1",

    "validation_macro_f1": float(study_xgb.best_value),

    "final_test": {
        "accuracy": float(final_metrics["Accuracy"]),
        "precision_macro": float(final_metrics["Precision_Macro"]),
        "recall_macro": float(final_metrics["Recall_Macro"]),
        "f1_macro": float(final_metrics["F1_Macro"]),
        "f1_weighted": float(final_metrics["F1_Weighted"]),
        "ordinal_mae": float(final_metrics["Ordinal_MAE"])
    },

    "data": {
        "development_rows": int(len(X_dev_final)),
        "final_test_rows": int(len(X_test_final)),
        "model_features": int(X_dev_final.shape[1])
    }
}

s3.put_object(
    Bucket=bucket,
    Key="evaluation/xgboost_final_test_metrics.json",
    Body=json.dumps(evaluation_summary, indent=4),
    ContentType="application/json"
)


# -----------------------------
# 2. Confusion matrix
# -----------------------------

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual_50-100",
        "Actual_100-150",
        "Actual_150-200",
        "Actual_200-250"
    ],
    columns=[
        "Pred_50-100",
        "Pred_100-150",
        "Pred_150-200",
        "Pred_200-250"
    ]
)

s3.put_object(
    Bucket=bucket,
    Key="evaluation/xgboost_final_confusion_matrix.csv",
    Body=cm_df.to_csv(),
    ContentType="text/csv"
)


# -----------------------------
# 3. Best Optuna parameters
# -----------------------------

s3.put_object(
    Bucket=bucket,
    Key="evaluation/xgboost_best_params.json",
    Body=json.dumps(
        study_xgb.best_params,
        indent=4
    ),
    ContentType="application/json"
)


print("Saved:")
print("s3://krushang-beverage-ml-2026/evaluation/xgboost_final_test_metrics.json")
print("s3://krushang-beverage-ml-2026/evaluation/xgboost_final_confusion_matrix.csv")
print("s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json")

Saved:
s3://krushang-beverage-ml-2026/evaluation/xgboost_final_test_metrics.json
s3://krushang-beverage-ml-2026/evaluation/xgboost_final_confusion_matrix.csv
s3://krushang-beverage-ml-2026/evaluation/xgboost_best_params.json
